# Scaling Laws and Compute Budgets

> MoE separates total parameters from per-token compute, but every architecture must allocate budget among parameters, data, and compute. Chinchilla used 70B parameters while Gopher used 280B; with similar training compute, the smaller model trained on more data performed better. Under a fixed budget, the **parameter-to-data ratio** matters more than model size alone.
>
> We build three connected ledgers: an **outcome ledger** for power-law relationships among parameters $N$, data $D$, and loss; a **compute ledger** using $C\approx6ND$ to estimate FLOPs and GPU-hours; and a **memory ledger** for parameters, optimizer state, and activations.
>
> Kaplan, Chinchilla, and LLaMA optimize different allocations, while µP transfers hyperparameters tuned on small models to larger ones. A viable plan must satisfy quality, compute, and hardware constraints simultaneously.

Training scale can increase through more parameters $N$ or more tokens $D$. Both consume compute, roughly $C\approx6ND$. At fixed $C$, increasing one reduces the affordable amount of the other. **Scaling Laws** use empirical curves to describe the relationship among $N$, $D$, $C$, and loss.


## 1. Power Laws and Diminishing Returns

Consider learning vocabulary. The first 100 common words cover a large fraction of everyday text; moving to 1,000 helps substantially; 10,000 helps further; and 100,000 brings a smaller marginal gain. Word frequency follows a power law, so each tenfold effort yields less improvement.

Language-model loss behaves similarly. A common form is $L(N)=aN^{-b}+c$: $a$ sets scale, $b$ controls decay, and $c$ is an irreducible floor. Every tenfold increase in $N$ multiplies the reducible part $L-c$ by the fixed factor $10^{-b}<1$. Diminishing returns are encoded in the curve.


In [ ]:
# === Hand-calculate and plot a power law: how much does loss fall when the model grows 10x? ===
a, b, c = 5.0, 0.05, 2.0   # illustrative fitted constants

# First calculate with concrete numbers to see the loss reduction at every 10x increase
sizes = [1, 10, 100, 1000, 10000, 100000]   # millions of parameters
labels = ["1M", "10M", "100M", "1B", "10B", "100B"]

print(f"Loss = {a} × N^(-{b}) + {c}")
print(f"{'size':>6s} {'loss':>7s} {'reducible':>10s} {'vs prev':>8s}")
print("-" * 36)
prev = None
for s, lab in zip(sizes, labels):
    N = s * 1e6
    loss = a * N ** (-b) + c
    gap = loss - c                          # reducible part = Loss - c
    if prev:
        print(f"{lab:>6s} {loss:>7.3f} {gap:>10.3f} {gap/prev*100:>7.1f}%")
    else:
        print(f"{lab:>6s} {loss:>7.3f} {gap:>10.3f}       —")
    prev = gap

print()
print("Key observation: each tenfold scale increase leaves about 10^(-0.05), or 89.1%, of the reducible term.")
print("          loss falls from 4.5 to 3.4; at later scales, a tenfold increase buys only about 0.2")

# Then plot the curve to see diminishing returns directly
import matplotlib.pyplot as plt
import numpy as np

N_range = np.logspace(6, 11, 300)
loss_range = a * N_range ** (-b) + c

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(N_range, loss_range, color='#2563eb', linewidth=2.2, label='Loss = a·N⁻ᵇ + c')
ax.axhline(c, color='#dc2626', linestyle='--', linewidth=1.5, label=f'floor c = {c}')

# Mark the hand-calculated discrete points
for s, lab in zip(sizes, labels):
    N = s * 1e6
    loss = a * N ** (-b) + c
    ax.plot(N, loss, 'o', color='#1e293b', markersize=4)
    ax.annotate(lab, (N, loss), textcoords="offset points", xytext=(0, 9),
                fontsize=8, ha='center')

ax.set_xscale('log')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Loss')
ax.set_title('Power Law: bigger model, smaller gain')
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
ax.set_ylim(2.0, 5.0)
plt.tight_layout()
plt.show()


With a fixed budget such as $10^{21}$ FLOPs, what model and dataset minimize loss?

Kaplan et al. (2020) fitted models from 1M to 1B parameters and estimated:

```text
N_opt proportional to C^0.73
D_opt proportional to C^0.27
```

Doubling compute would multiply model size by $2^{0.73}\approx1.66$ but data by only $2^{0.27}\approx1.21$. GPT-3 reflects this recipe: 175B parameters trained on 300B tokens, only about 1.7 tokens per parameter.


In [ ]:
# === Kaplan calculation and plot: when budget doubles, how quickly do model and data grow? ===
N_ref, D_ref = 1e9, 20e9          # reference: when C = 6ND ≈ 1.2e20, N=1B and D=20B
C_ref = 6 * N_ref * D_ref

print(f"Reference: N={N_ref/1e9:.0f}B, D={D_ref/1e9:.0f}B tokens (C≈{C_ref:.1e} FLOPs)")
print()
print(f"{'Budget':>7s} {'Model (×0.73)':>13s} {'Data (×0.27)':>13s} {'D/N':>7s}")
print("-" * 46)

mults = [1, 2, 4, 8, 10, 100]
Ns = [N_ref * m ** 0.73 for m in mults]
Ds = [D_ref * m ** 0.27 for m in mults]
for m, N, D in zip(mults, Ns, Ds):
    print(f"{m:>6d}× {N/1e9:>9.1f}B {D/1e9:>9.1f}B {D/N:>6.1f}x")

print()
print("Key observation:")
print("  1. At 100x budget, model size grows by 100^0.73 ≈ 29x, while data grows only 100^0.27 ≈ 3.5x")
print("  2. D/N keeps falling: in Kaplan's view, large models inherently use less data per parameter")

# Plot the curves: on log-log axes, slopes 0.73 and 0.27 diverge steadily
import numpy as np
import matplotlib.pyplot as plt

mult_range = np.logspace(0, 2, 200)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(mult_range, N_ref * mult_range ** 0.73, color='#2563eb',
        linewidth=2.2, label='N ∝ C^0.73 (model)')
ax.plot(mult_range, D_ref * mult_range ** 0.27, color='#dc2626',
        linewidth=2.2, label='D ∝ C^0.27 (data)')
ax.plot(mults, Ns, 'o', color='#2563eb', markersize=5)
ax.plot(mults, Ds, 'o', color='#dc2626', markersize=5)
ax.annotate('x29', (100, Ns[-1]), textcoords="offset points", xytext=(4, 6),
            fontsize=9, color='#2563eb')
ax.annotate('x3.5', (100, Ds[-1]), textcoords="offset points", xytext=(4, -12),
            fontsize=9, color='#dc2626')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Compute budget multiplier (x)')
ax.set_ylabel('Size (params / tokens)')
ax.set_title('Kaplan: budget favors model over data')
ax.legend(loc='center left')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Chinchilla Scaling Laws

Hoffmann et al. (2022) repeated the study and reached a different result.

### 3.1 Limits of Kaplan Scaling Laws

Kaplan used learning-rate schedules that were not aligned to each run's actual length, systematically underestimating small models trained longer. Different fitted functional forms also favored larger models.

DeepMind trained about 400 models from 70M to 16B parameters, aligned schedules to training length, and cross-checked fixed-model, fixed-data, and direct loss-surface methods.

### 3.2 Model and Data Scale

Their compute-optimal result was approximately:

```text
D_opt about 20 x N_opt
N and D both proportional to C^0.5
```

When compute doubles, both model size and data grow by $2^{0.5}\approx1.41$.


In [ ]:
# === What do two recipes buy with the same budget? ===
C = 1e23                              # roughly GPT-3-scale training compute
mult = C / C_ref

N_k, D_k = N_ref * mult ** 0.73, D_ref * mult ** 0.27   # Kaplan recipe
N_c, D_c = N_ref * mult ** 0.5,  D_ref * mult ** 0.5    # Chinchilla recipe

print(f"Compute budget: {C:.0e} FLOPs")
print()
print(f"{'Recipe':<12s} {'Model':>6s} {'Data':>13s} {'D/N':>6s}")
print("-" * 42)
print(f"{'Kaplan':<12s} {N_k/1e9:>4.0f}B {D_k/1e9:>8.0f}B tok {D_k/N_k:>5.1f}x")
print(f"{'Chinchilla':<12s} {N_c/1e9:>4.0f}B {D_c/1e9:>8.0f}B tok {D_c/N_c:>5.1f}x")
print()
print("Key observation:")
print(f"  1. Kaplan's model is {N_k/N_c:.1f}x larger, with only {D_k/D_c*100:.0f}% as much data as Chinchilla")
print("  2. Measured loss is lower under the Chinchilla recipe, overturning the Kaplan allocation")

# Plot side by side to make the two purchases under the same budget visible
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
recipes = [
    ('Kaplan',   N_k, D_k, '#2563eb'),
    ('Chinchilla', N_c, D_c, '#16a34a'),
]
for ax, (name, N, D, color) in zip(axes, recipes):
    bars = ax.bar(['model\nparams', 'train\ntokens'],
                  [N / 1e9, D / 1e9], color=color, width=0.55)
    ax.bar_label(bars, fmt='%.0fB', fontsize=9)
    ax.set_title(f'{name}: D/N = {D/N:.0f}x')
    ax.set_ylabel('Billions')
    ax.grid(True, axis='y', alpha=0.3)
fig.suptitle('Same budget, two recipes: Kaplan buys model, Chinchilla buys data',
             fontsize=11)
plt.tight_layout()
plt.show()


### 3.3 Gopher and Chinchilla

DeepMind compared two recipes at nearly identical compute:

| Model | Parameters | Training data | Recipe |
|:---|---:|---:|:---|
| Gopher | 280B | 300B tokens | Large model, less data |
| Chinchilla | 70B | 1.4T tokens | Smaller model, 4.7x more data |

Chinchilla outperformed Gopher on almost every benchmark despite having one quarter as many parameters. Many 2020–2022 models were undertrained relative to their size.


## 4. LLaMA and Over-Training

Chinchilla optimizes the cost of one training run. A deployed model may serve millions of requests, and inference cost depends strongly on parameter count. Training a smaller model on far more than 20 tokens per parameter can cost more once but save repeatedly during inference. This is **over-training** relative to the compute-optimal training recipe.

LLaMA 3 8B illustrates the shift. Chinchilla suggests roughly 160B tokens; the model used 15T, about 94 times as many. The resulting 8B model is unusually capable for its size while inference is far cheaper than a 70B model.


In [ ]:
# === D/N in real models: industry moved far beyond 20 ===
models = [
    ("GPT-3 (2020)",       175,   300),
    ("Chinchilla (2022)",   70,  1400),
    ("LLaMA 7B (2023)",      7,  1000),
    ("LLaMA 2 7B (2023)",    7,  2000),
    ("LLaMA 3 8B (2024)",    8, 15000),
    ("LLaMA 3 70B (2024)",  70, 15000),
    ("DeepSeek-V2 (2024)", 236,  8100),
]

print(f"{'Model':<20s} {'Params':>6s} {'Tokens':>8s} {'D/N':>7s}")
print("-" * 46)
for name, p, d in models:
    print(f"{name:<20s} {p:>5d}B {d:>6d}B {d/p:>6.1f}x")

print()
print("Key observation:")
print("  1. In 2020, each GPT-3 parameter saw only 1.7 Tokens, clearly undertrained")
print("  2. In 2024, each parameter in an 8B model saw 1,875 Tokens, 94x the Chinchilla optimum")
print("  3. LLaMA 3 trains both 8B and 70B on the same 15T Tokens: the same data for every size,")
print("     with smaller, cheaper-to-infer models receiving much greater overtraining")
print()
print("Note: DeepSeek-V2 is MoE; 236B is total parameters, while 21B active parameters gives 386x")

# Plot by year: D/N rises from 2 to 1,875, far beyond the nominal optimum of 20
import matplotlib.pyplot as plt

years = [int(n.split('(')[1][:-1]) for n, p, d in models]
dn = [d / p for n, p, d in models]

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.scatter(years, dn, s=70, color='#2563eb', zorder=3)
for (name, p, d), y, r in zip(models, years, dn):
    short = name.split(' (')[0]
    ax.annotate(f"{short}\n{d/p:.0f}x", (y, r), textcoords="offset points",
                xytext=(7, 4), fontsize=8)
ax.axhline(20, color='#dc2626', linestyle='--', linewidth=1.5)
ax.text(2020.05, 24, 'Chinchilla optimal ~20x', color='#dc2626', fontsize=9)
ax.set_yscale('log')
ax.set_ylim(1, 5000)
ax.set_xlabel('Year')
ax.set_ylabel('Tokens per parameter (D/N)')
ax.set_title('Real models: D/N keeps climbing past the Chinchilla line')
ax.set_xticks([2020, 2021, 2022, 2023, 2024])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. µP and Hyperparameter Transfer

Learning rate and initialization must be chosen before training a 70B model, but grid search at that size is prohibitive. Tuning on a small proxy often fails because activation and gradient scales change with width.

**µP** (Maximal Update Parameterization, Tensor Programs V, 2022) specifies how initialization variance and learning rate scale for different parameter types so that one update changes outputs by a width-independent amount. Hyperparameters found on a small model can then transfer zero-shot to a larger model, reducing tuning compute dramatically.

Libraries such as `microsoft/mup` and training platforms package these rules. A small validation sweep at target scale remains prudent: µP improves transfer reliability but does not guarantee it.


In [ ]:
# === Why tuning feel drifts with width: begin with activation values ===
import numpy as np

rng = np.random.default_rng(42)

widths = [64, 256, 1024, 4096, 16384]
std_naive = []
std_scaled = []

print(f"{'fan_in width':>10s} {'output std for N(0,1)':>20s} {'output std for N(0,1/fan_in)':>26s}")
print("-" * 60)
for width in widths:
    x = rng.normal(size=width)                      # input dimension = layer width
    x = x / x.std()                                 # fix empirical variance at 1 for comparison
    W_naive = rng.normal(0, 1.0, (1024, width))     # variance does not scale with width
    W_scaled = rng.normal(0, width ** -0.5, (1024, width))  # variance = 1/fan_in
    s1 = (W_naive * x).sum(axis=1).std()
    s2 = (W_scaled * x).sum(axis=1).std()
    std_naive.append(s1)
    std_scaled.append(s2)
    print(f"{width:>10d} {s1:>20.1f} {s2:>26.3f}")

print()
print("Key observation:")
print("  1. Without variance scaling, 64x width produces 8x output std (square-root relation), so activation scale drifts")
print("  2. With variance 1/fan_in, output std stays near 1; this is the scale preserved by the rule")
print("  3. µP coordinates both initialization variance and learning-rate scaling,")
print("     Decoupling width from optimal hyperparameters lets small-model tuning transfer to larger models")

# Plot two curves: unscaled grows rapidly while scaled stays near 1
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(widths, std_naive, 'o-', color='#dc2626', linewidth=2,
        label='no scaling: W ~ N(0, 1)')
ax.plot(widths, std_scaled, 'o-', color='#2563eb', linewidth=2,
        label='scaled: W ~ N(0, 1/fan_in)')
ax.axhline(1.0, color='#9ca3af', linestyle=':', linewidth=1.2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Layer width (fan_in)')
ax.set_ylabel('Output std after Linear')
ax.set_title('Init scaling keeps activation scale stable across widths')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Training FLOPs

Once the recipe is chosen, estimate total compute with $C\approx6ND$. For each token, a forward pass costs roughly $2N$ FLOPs because a multiply-add counts as two operations. Backward calculates input and weight gradients, roughly $4N$. The total is therefore about $6N$ FLOPs per token, or $6ND$ over $D$ tokens.

Attention scores and Embedding operations are smaller-order terms absorbed by the approximation. Appendix B develops a layer-by-layer ledger; $6ND$ is the first planning estimate.


In [ ]:
# === FLOPs calculation: LLaMA 7B plus 1T Tokens ===
N = 7e9        # 7B parameters
D = 1e12       # 1T tokens

flops_per_token = 6 * N
total_flops = flops_per_token * D

print(f"Per Token: 6 × {N:.1e} = {flops_per_token:.1e} FLOPs")
print(f"A total of {D:.0e} Tokens: {total_flops:.1e} FLOPs")
print()
print("Key observation:")
print(f"  1. A 7B model performs {flops_per_token/1e9:.0f}B operations for each Token")
print(f"  2. One 1,000-Token sample costs {flops_per_token * 1000 / 1e12:.0f} TFLOPs")
print("  3. This scale is hard to feel directly; next convert it into GPUs and days")


## 7. Training Time and Cost

Engineering plans convert FLOPs into GPU-hours. Effective throughput is below hardware peak because communication, data loading, and memory effects leave compute units idle. MFU (Model FLOPs Utilization) commonly falls around 40–60%. GPU peak, achieved MFU, and rental price must all enter the estimate.


In [ ]:
# === Effective compute of common training GPUs, illustrative estimates ===
# Rental rates change quickly by region, supply, demand, and contract length; these numbers only demonstrate the calculation.
# Written in 2026-08; recalculate real budgets using current quotes.
gpus = [
    ("A100 80GB",  312, 0.50, 2.0),   # (name, peak TFLOPS, utilization, illustrative $/hour)
    ("H100 80GB",  990, 0.50, 3.5),
    ("H200 141GB", 990, 0.50, 4.5),
    ("RTX 4090",   165, 0.40, 0.5),
]

print(f"{'GPU':<14s} {'Peak':>6s} {'Util.':>6s} {'Effective':>8s} {'Example $/h':>8s}")
print("-" * 48)
for name, peak, util, price in gpus:
    print(f"{name:<14s} {peak:>4d}TF {util*100:>5.0f}% {peak*util:>6.0f}TF"
          f"   ${price:.1f}/h")

print()
print("Key observation: effective H100 throughput is about three times A100 while hourly rental is less than twice as high,")
print("          When the budget allows, renting H-series GPUs can be more economical, subject to interconnect and memory.")


In [ ]:
# === GPU-hours calculation: LLaMA 7B plus 1T Tokens ===
total_flops = 6 * 7e9 * 1e12

print(f"Total FLOPs = {total_flops:.1e}")
print()
print(f"{'GPU':<14s} {'GPU-hours':>12s} {'1-GPU days':>10s} {'256-GPU days':>10s}")
print("-" * 52)
for name, peak, util, price in gpus:
    eff = peak * util                       # effective compute in TFLOPS
    hours = total_flops / (eff * 1e12 * 3600)
    print(f"{name:<14s} {hours:>12,.0f} {hours/24:>10,.0f} {hours/256/24:>10.1f}")

print()
print("Key observation:")
print("  1. GPU-hours depend only on total compute, not the number of GPUs:")
print("     256 GPUs for one day ≈ one GPU for 256 days; the total is identical")
print("  2. More GPUs buy time, not less total computation")


In [ ]:
# === Training bills for major models, plotted with an A100-based estimate ===
a100_eff = 312 * 0.5        # A100 effective compute: 156 TFLOPS
a100_price = 2.0            # illustrative rental rate: $2/hour

models = [
    ("GPT-3 175B",           175e9,  300e9),
    ("Chinchilla 70B",        70e9, 1400e9),
    ("LLaMA 2 70B",           70e9, 2000e9),
    ("LLaMA 3 8B",             8e9,   15e12),
    ("LLaMA 3 70B",           70e9,   15e12),
    ("DeepSeek-V3 (37B active)", 37e9, 14.8e12),
]

print(f"{'Model':<24s} {'Params':>5s} {'Tokens':>7s} {'FLOPs':>8s} "
      f"{'GPU-hours':>11s} {'2048 GPU-days':>9s} {'Cost':>6s}")
print("-" * 78)
costs_musd = []
names = []
for name, p, d in models:
    flops = 6 * p * d
    hours = flops / (a100_eff * 1e12 * 3600)
    cost_m = hours * a100_price / 1e6
    costs_musd.append(cost_m)
    names.append(name.split(' (')[0] if 'active' in name else name)
    print(f"{name:<24s} {p/1e9:>4.0f}B {d/1e9:>5.0f}B {flops:>8.1e} "
          f"{hours:>11,.0f} {hours/2048/24:>9.1f} ${cost_m:>5.1f}M")

print()
print("Method: GPU-hours = FLOPs / (156 TFLOPS × 3600); cost = GPU-hours × $2")
print("Key observation:")
print("  1. DeepSeek-V3 is MoE: N in 6ND uses 37B active parameters, not 671B total")
print("  2. This is a lower bound for one run; failed restarts, evaluation, and ablations add more")

# Plot training cost: within four years, estimates rise from $4M to over $50M
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = ['#94a3b8'] * len(names)
colors[-1] = '#2563eb'
bars = ax.barh(names[::-1], costs_musd[::-1], color=colors[::-1], height=0.6)
ax.bar_label(bars, fmt='$%.1fM', fontsize=9, padding=3)
ax.set_xlabel('Estimated training cost (million USD, A100 @ $2/h)')
ax.set_title('Training bills keep climbing')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Training Memory

For full mixed-precision training with AdamW, each parameter typically requires:

```text
parameter (FP16/BF16)      2 bytes
gradient (FP16/BF16)       2 bytes
Adam first moment m (FP32) 4 bytes
Adam second moment v       4 bytes
FP32 master weight         4 bytes
fixed total               16 bytes per parameter
activations                depend on batch, sequence, and layers
```

A rough planning rule reserves about **20 bytes per parameter**, including an activation allowance. A 7B model therefore needs roughly 140 GB before accounting for implementation details, requiring multiple 80 GB GPUs for full training.


In [ ]:
# === Memory calculation: reserve 20 bytes per parameter for full training ===
P = 7e9

parts = [
    ("Parameters (FP16/BF16)",   2 * P),
    ("Gradients (FP16/BF16)",   2 * P),
    ("Adam m (FP32)",      4 * P),
    ("Adam v (FP32)",      4 * P),
    ("Master weights (FP32)", 4 * P),
]
fixed = sum(n_bytes for _, n_bytes in parts)

print("Model output:")
for name, n_bytes in parts:
    print(f"  {name:<20s} {n_bytes/1e9:>6.1f} GB")
print(f"  {'Subtotal':<20s} {fixed/1e9:>6.1f} GB (= 16 × P bytes)")
print()

print(f"{'Model':<8s} {'Fixed':>9s} {'With activations':>10s} {'A100 80G':>9s}")
print("-" * 42)
for name, p in [("1.5B", 1.5e9), ("7B", 7e9), ("13B", 13e9),
                ("70B", 70e9), ("175B", 175e9)]:
    fixed_i, total_i = 16 * p, 20 * p
    print(f"{name:<8s} {fixed_i/1e9:>7.0f} GB {total_i/1e9:>7.0f} GB"
          f" {total_i/80e9:>7.1f} GPUs")

print()
print("Key observation:")
print("  1. Weights themselves are only 2/16; gradients and optimizer state occupy the other 14 parts")
print("  2. Twenty bytes per parameter is a rough starting point for full AdamW training;")
print("     ZeRO/FSDP sharding, 8-bit optimizers, and activation recomputation change it")
print("  3. Inference has no gradients or optimizer; memory is mainly weights plus KV Cache, so do not use this table")


## 9. Comparing Three Scaling Paradigms

| Year and recipe | Allocation | Representative model | Objective |
|:---|:---|:---|:---|
| 2020 Kaplan | $N\propto C^{0.73}$, $D\propto C^{0.27}$ | GPT-3 175B + 300B tokens | Training loss |
| 2022 Chinchilla | $N,D\propto C^{0.5}$, $D/N\approx20$ | Chinchilla 70B + 1.4T tokens | Training loss |
| 2023+ LLaMA | $D/N$ far above 20 | LLaMA 3 8B + 15T tokens | Inference-aware total cost |

Kaplan and Chinchilla ask how to spend training compute; LLaMA changes the objective to training plus lifetime inference. The conclusions differ because the optimized objective differs.


## Summary

Confirm you understand these (check in order):

1. ✅ Power law: Loss ∝ N^(-b), diminishing returns—each doubling gives less improvement
2. ✅ Kaplan: prioritize model size, D grows slower than N
3. ✅ Chinchilla: model and data equally important, D/N ≈ 20
4. ✅ Overtraining: inference cost > training cost → small model with more data is more cost-effective
5. ✅ µP: enables hyperparameter sharing across model sizes, reducing tuning cost
6. ✅ FLOPs formula: C ≈ 6PD (forward 2P + backward 4P per token)
7. ✅ GPU-hours = Total FLOPs ÷ (effective compute × 3600); GPU-hours are independent of GPU count
8. ✅ Memory estimation: 20 bytes/param is a rough starting point for AdamW full training, not a fixed formula
9. ✅ Real memory also depends on ZeRO/FSDP, activation checkpointing, optimizer, batch, and seq_len

**One-sentence summary**: Scaling laws tell you the optimal ratio (theory), while FLOPs/memory/GPU-hours estimation tells you whether it is feasible and how much it costs (engineering). The 2024 consensus is—small model + massive data, optimizing for inference cost.

## Exercises

> You can ask an AI to explain the ideas or check your direction, but don't have it "solve the exercise" for you.

**Exercise 1: Convert FLOPs to PFLOPs-days**

For LLaMA 7B trained on 1T tokens, calculate $C\approx6ND$, then convert using 1 PFLOPs-day $=10^{15}\times86400$ FLOPs.

Hint: calculate $6\times7\times10^9\times10^{12}$ and divide by $8.64\times10^{19}$.


In [ ]:
# Exercise 1: FLOPs to PFLOPs-days
N = 7e9      # 7B parameters
D = 1e12     # 1T tokens

# TODO: calculate total FLOPs, C ≈ 6ND
total_flops = None

# TODO: convert to PFLOPs-days; 1 PFLOPs-day = 1e15 × 86400 FLOPs
pflops_days = None

# Uncomment the checks below after filling the answers
# assert total_flops == 6 * N * D
# assert abs(pflops_days - 6 * N * D / (1e15 * 86400)) < 0.01
# print(f"✅ Exercise 1 passed: {pflops_days:.0f} PFLOPs-days")
# print("   Equivalent to about 1.5 days at full load on 2,048 A100s delivering 156 TFLOPS each.")


**Exercise 2: Growth When Compute Doubles**

Calculate the model-size multiplier when compute doubles under Kaplan ($N\propto C^{0.73}$) and Chinchilla ($N\propto C^{0.5}$).

Hint: $N_{new}/N_{old}=2^{exponent}$.


In [ ]:
# Exercise 2: model growth when budget doubles
ratio = 2    # double the budget

# TODO: Kaplan model-growth ratio, N ∝ C^0.73
kaplan_ratio = None

# TODO: Chinchilla model-growth ratio, N ∝ C^0.5
chinchilla_ratio = None

# Uncomment the checks below after filling the answers
# assert abs(kaplan_ratio - 2 ** 0.73) < 0.01
# assert abs(chinchilla_ratio - 2 ** 0.5) < 0.01
# print(f"✅ Exercise 2 passed: Kaplan ×{kaplan_ratio:.2f}, Chinchilla ×{chinchilla_ratio:.2f}")
# print("   With the same doubled budget, Kaplan allocates more toward a larger model.")


**Exercise 3: Fixed Memory for Full 7B Training**

Using 2 bytes for parameters, 2 for gradients, 4 each for Adam $m$ and $v$, and 4 for master weights, calculate fixed memory for 7B parameters.

Hint: multiply 16 bytes by $7\times10^9$ and convert units.


In [ ]:
# Exercise 3: fixed memory for full training of a 7B model
P = 7e9

# TODO: fixed overhead in bytes: 2P + 2P + 4P + 4P + 4P
fixed_bytes = None

# TODO: convert to GB
fixed_gb = None

# Uncomment the checks below after filling the answers
# assert fixed_bytes == 16 * P
# assert abs(fixed_gb - 16 * P / 1e9) < 0.1
# print(f"✅ Exercise 3 passed: fixed overhead {fixed_gb:.0f} GB")
# print("   Weights alone use 14 GB; gradients and optimizer states use the remaining 98 GB,")
# print("   which is why a 7B model does not fit on one A100 80GB for full training.")


## References

- Kaplan et al., [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361), 2020 — the original model-heavy allocation result
- Hoffmann et al., [Training Compute-Optimal Large Language Models](https://arxiv.org/abs/2203.15556), 2022 — Chinchilla's balanced model/data allocation
- Yang et al., [Tensor Programs V](https://arxiv.org/abs/2203.03466), 2022 — zero-shot hyperparameter transfer with µP
- Meta, [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783), 2024 — extensive token over-training of 8B and 70B models
- [microsoft/mup](https://github.com/microsoft/mup) — reference implementation of µP
